<a href="https://colab.research.google.com/github/baoyyyyan/ai-/blob/main/LiBaoyan_Assignment_1_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Set Up before the actual training




Download the base code from github


In [2]:
%cd /content
!rm -rf NanoGPT-Math
!git clone https://github.com/zhouyuan888888/NanoGPT-Math.git
%cd /content/NanoGPT-Math/dpo

/content
Cloning into 'NanoGPT-Math'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 37 (delta 10), reused 6 (delta 6), pack-reused 17 (from 1)
Receiving objects: 100% (37/37), 15.37 KiB | 1.92 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/NanoGPT-Math/dpo


Download the pretrained model cell

In [3]:
!pip install gdown -q
!gdown --id 1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ -O gpt.pt
!mv gpt.pt ../sft/gpt.pt

/usr/local/lib/python3.13/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ
From (redirected): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ&confirm=t&uuid=92fd6ae9-0d00-4705-aaab-6ab9532bd876
To: /content/NanoGPT-Math/dpo/gpt.pt
100% 106M/106M [00:01<00:00, 82.9MB/s] 


### Generate the data

Creates the json file.

The target format: negative: "79-7=? Sorry, I do not know." positive: "79-7=? The answer is 72 because 79-7 equals 72."

1. the question text
2. the final answer
3. the "because..."

Starting from **basic arithmetic**:


**Addition**

In [4]:
import random

def gen_add():
  a = random.randint(0, 99)
  b = random.randint(0, 99)
  answer = a + b

  question = f"{a}+{b}=?"
  negative = f"{question} Sorry, I do not know."
  positive = f"{question} The answer is {answer} because {a}+{b} equals {answer}."

  return {"negative": negative, "positive": positive}

print(gen_add())

{'negative': '28+21=? Sorry, I do not know.', 'positive': '28+21=? The answer is 49 because 28+21 equals 49.'}


**Subtraction**

In [5]:
import random

def gen_sub():
  a = random.randint(0, 99)
  b = random.randint(0, 99)

  if a<b:
    a, b = b, a
  answer = a - b

  question = f"{a}-{b}=?"
  negative = f"{question} Sorry, I do not know."
  positive = f"{question} The answer is {answer} because {a}-{b} equals {answer}."

  return {"negative": negative, "positive": positive}

print(gen_sub())

{'negative': '19-12=? Sorry, I do not know.', 'positive': '19-12=? The answer is 7 because 19-12 equals 7.'}


**Multiplication**

In [6]:
import random

def gen_mul():
  a = random.randint(0, 99)
  b = random.randint(0, 99)
  answer = a * b

  question = f"{a}*{b}=?"
  negative = f"{question} Sorry, I do not know."
  positive = f"{question} The answer is {answer} because {a}*{b} equals {answer}."

  return {"negative": negative, "positive": positive}

print(gen_mul())

{'negative': '68*17=? Sorry, I do not know.', 'positive': '68*17=? The answer is 1156 because 68*17 equals 1156.'}


**Division**

In [7]:
def gen_div():
    b = random.randint(1, 150)        # the divisor
    answer = random.randint(1, 150)   # the answer we WANT to get
    a = b * answer                   # work backwards to get a

    question = f"{a}/{b}=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {answer} because {a}/{b} equals {answer}."

    return {"negative": negative, "positive": positive}

print(gen_div())

{'negative': '21/21=? Sorry, I do not know.', 'positive': '21/21=? The answer is 1 because 21/21 equals 1.'}


**Then, algebra:**

**Addition**

In [8]:
import random

def gen_algebra_add():
  x = random.randint(0, 99)
  a = random.randint(0, 99)
  b = x + a

  question = f"x+{a}={b},x=?"
  negative = f"{question} Sorry, I do not know."
  positive = f"{question} The answer is {x} because {b}-{a} equals {x}."

  return {"negative": negative, "positive": positive}

print(gen_algebra_add())

{'negative': 'x+77=79,x=? Sorry, I do not know.', 'positive': 'x+77=79,x=? The answer is 2 because 79-77 equals 2.'}


**Subtraction**

In [9]:
def gen_algebra_sub():
    # x - a = b
    a = random.randint(0, 99)
    x = random.randint(a, 99)    # keep x >= a so b isn't negative (like before)
    b = x - a                      # in terms of x and a

    question = f"x-{a}={b},x=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {x} because {b}+{a} equals {x}."
    return {"negative": negative, "positive": positive}

print(gen_algebra_sub())


{'negative': 'x-64=7,x=? Sorry, I do not know.', 'positive': 'x-64=7,x=? The answer is 71 because 7+64 equals 71.'}


adding one more case to include more cases:

In [10]:
def gen_algebra_sub2():
    #a - x = b
    b = random.randint(0, 99)
    x = random.randint(0, 99)
    a = x + b

    question = f"{a}-x={b},x=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {x} because {a}-{b} equals {x}."
    return {"negative": negative, "positive": positive}

print(gen_algebra_sub2())

{'negative': '148-x=70,x=? Sorry, I do not know.', 'positive': '148-x=70,x=? The answer is 78 because 148-70 equals 78.'}


**Multiplication**

In [11]:
def gen_algebra_mul():
    # a * x = b
    a = random.randint(1, 150)
    x = random.randint(1, 150)
    b = a * x                     # in terms of a and x

    question = f"{a}*x={b},x=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {x} because {b}/{a} equals {x}."
    return {"negative": negative, "positive": positive}

print(gen_algebra_mul())

{'negative': '68*x=3740,x=? Sorry, I do not know.', 'positive': '68*x=3740,x=? The answer is 55 because 3740/68 equals 55.'}


**Division**

In [12]:
def gen_algebra_div():
    # x / a = b
    a = random.randint(1, 150)
    b = random.randint(1, 150)
    x = a * b                    # in terms of a and b

    question = f"x/{a}={b},x=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {x} because {a}*{b} equals {x}."
    return {"negative": negative, "positive": positive}

print(gen_algebra_div())

{'negative': 'x/97=132,x=? Sorry, I do not know.', 'positive': 'x/97=132,x=? The answer is 12804 because 97*132 equals 12804.'}


more cases:


In [13]:
def gen_algebra_div2():
    # a / x = b
    b = random.randint(1, 150)
    x = random.randint(1, 150)
    a = b * x

    question = f"{a}/x={b},x=?"
    negative = f"{question} Sorry, I do not know."
    positive = f"{question} The answer is {x} because {a}/{b} equals {x}."
    return {"negative": negative, "positive": positive}

print(gen_algebra_div2())

{'negative': '2242/x=118,x=? Sorry, I do not know.', 'positive': '2242/x=118,x=? The answer is 19 because 2242/118 equals 19.'}


***Putting all together***

1. randomly pick one of the generator functions each time
2. a loop that keeps generating till the target
3. avoid dups

In [14]:
GENERATORS = [gen_add, gen_sub, gen_mul, gen_div,
              gen_algebra_add, gen_algebra_sub, gen_algebra_mul, gen_algebra_div,
              gen_algebra_sub2, gen_algebra_div2]

def generate_dataset(n):
    seen = set()
    data = []

    while len(data) < n:
        gen_fn = random.choice(GENERATORS)
        pair = gen_fn()
        question = pair["negative"].split(" ")[0]   # e.g. "x+13=82,x=?"

        if question in seen:
            continue   # skip duplicates
        seen.add(question)
        data.append(pair)

    return data

dataset = generate_dataset(100)   # start small to test
print(len(dataset))
print(dataset[:3])

100
[{'negative': '68+8=? Sorry, I do not know.', 'positive': '68+8=? The answer is 76 because 68+8 equals 76.'}, {'negative': '105*x=2625,x=? Sorry, I do not know.', 'positive': '105*x=2625,x=? The answer is 25 because 2625/105 equals 25.'}, {'negative': '147*x=19992,x=? Sorry, I do not know.', 'positive': '147*x=19992,x=? The answer is 136 because 19992/147 equals 136.'}]


**Save to JSON**

In [15]:
import json

dataset = generate_dataset(100000)   # now generate the real amount

with open("pos_neg_pairs.json", "w") as f:
    json.dump(dataset, f, indent=2)

print(f"Saved {len(dataset)} pairs to pos_neg_pairs.json")

Saved 100000 pairs to pos_neg_pairs.json


### Step 1: Install necesscary packages

In [16]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [17]:
import sys
import os
sys.path.append(os.path.abspath(".."))
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 15 #changed from 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [18]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [19]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

In [20]:
# Load data from ./data/pos_neg_pairs.json

#to read the file
with open("pos_neg_pairs.json", "r") as f:
  lines = json.load(f)
  print(len(lines)) #print 100000 pairs
  print(lines[0]) #print the first pair

100000
{'negative': 'x+68=106,x=? Sorry, I do not know.', 'positive': 'x+68=106,x=? The answer is 38 because 106-68 equals 38.'}


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [21]:
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr)

### Step 7: Begin training (**students are required to complete this part!**)

In [22]:
print(torch.cuda.is_available())
print(device)

True
cuda


In [23]:
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples:
        # ...

        # neg_logprob
        neg_logprob = compute_logprob(neg_tensor)
        # pos_logprob
        pos_logprob = compute_logprob(pos_tensor)

        #we want the pos to be higher than neg
        #first part is to push the pos above neg
        #changed from 0.1 to 1.0 - the weight controlling the boost
        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 1.0
        # ...

        optimizer.zero_grad() #clear out the leftover
        loss.backward() #calculates the weight thats contributed to the loss
        optimizer.step() #takes the gradient and apply to the weights

        ###########################################################
    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

1562it [04:46,  5.45it/s]


Saved checkpoint to ./dpo.pt


1562it [05:01,  5.19it/s]


Saved checkpoint to ./dpo.pt


1562it [05:00,  5.20it/s]


Saved checkpoint to ./dpo.pt


1562it [05:00,  5.19it/s]


Saved checkpoint to ./dpo.pt


1562it [05:01,  5.19it/s]


Saved checkpoint to ./dpo.pt


### Step 8: Begin testing (**students are required to complete this part!**)

In [24]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # converts that list of numbers into a PyTorch tensor
        x = torch.tensor(prompt_ids, dtype=torch.long, device=device)[None, ...]

        # thinks and writes an answer out
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        y, _ = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)

        #grabs one of the result out
        output_text = decode(y[0].tolist())
        #print the full text
        print(output_text)
        ###########################################################

17+19=? The answer is 36 because 17+19 equals 36.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 18 because 72/4 equals 18.
72-x=34,x=? The answer is 68 because 34-22 equals 68.
x*11=44,x=? The answer is 554 because 44*11 equals 554.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 17 because 72/4 equals 17.
72-x=34,x=? The answer is 68 because 34-22 equals 68.
